# **UNIT I:** **Running Total of Sales by Territory Over Time**

**Why should this proposition and the query be considered special?**

- It helps track whether sales are growing steadily or slowing down in specific territories by showing how sales accumulate over time in each region.

In [1]:
USE AdventureWorks2017;
GO

SELECT TerritoryID AS "Territorial Identification Number", OrderDate AS "Date of the Order", TotalDue AS "Total Due", SUM(TotalDue) OVER(PARTITION BY TerritoryID ORDER BY OrderDate ROWS UNBOUNDED PRECEDING) AS "Running Total"
FROM Sales.SalesOrderHeader
ORDER BY TerritoryID, OrderDate;

Commands completed successfully.

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.468

Territorial Identification Number,Date of the Order,Total Due,Running Total
1,2011-05-31 00:00:00.000,27510.4109,27510.4109
1,2011-05-31 00:00:00.000,16158.6961,43669.107
1,2011-05-31 00:00:00.000,807.2585,44476.3655
1,2011-05-31 00:00:00.000,9153.6054,53629.9709
1,2011-05-31 00:00:00.000,48204.0662,101834.0371
1,2011-05-31 00:00:00.000,3899.6756,105733.7127
1,2011-05-31 00:00:00.000,3756.989,109490.7017
1,2011-06-03 00:00:00.000,3953.9884,113444.6901
1,2011-06-05 00:00:00.000,3953.9884,117398.6785
1,2011-06-06 00:00:00.000,772.5036,118171.1821


# **UNIT II:** **Monthly Sales Comparison**

**Why should this proposition and the query be considered special?**

- It instantly reveals whether sales are improving or declining through monthly comparisons by using the LAG function to compare each month’s sales to the previous month.

In [2]:
USE AdventureWorks2017;
GO

SELECT YEAR(OrderDate) AS "Year of the Order", MONTH(OrderDate) AS "Month of the Order", SUM(TotalDue) AS "Monthly Sales", LAG(SUM(TotalDue)) OVER(ORDER BY YEAR(OrderDate), MONTH(OrderDate)) AS "Sales from the Previous Month"
FROM Sales.SalesOrderHeader
GROUP BY YEAR(OrderDate), MONTH(OrderDate);

Commands completed successfully.

(38 rows affected)

Total execution time: 00:00:00.052

Year of the Order,Month of the Order,Monthly Sales,Sales from the Previous Month
2011,5,567020.9498,NULL
2011,6,507096.469,567020.9498
2011,7,2292182.8828,507096.469
2011,8,2800576.1723,2292182.8828
2011,9,554791.6082,2800576.1723
2011,10,5156269.5291,554791.6082
2011,11,815313.0152,5156269.5291
2011,12,1462448.8986,815313.0152
2012,1,4458337.4444,1462448.8986
2012,2,1649051.9001,4458337.4444


# **UNIT III:** **Salesperson Percentage of Total Sales**

**Why should this proposition and the query be considered special?**

- It highlights top performers and their impact on overall revenue by calculating individual contributions as percentages of total company sales.

In [3]:
USE AdventureWorks2017;
GO

SELECT SalesPersonID AS "Identification Number of the Salesperson", SUM(TotalDue) AS "Personal Sales", 100.0 * SUM(TotalDue) / SUM(SUM(TotalDue)) OVER() AS "Percentage of the Sales"
FROM Sales.SalesOrderHeader
WHERE SalesPersonID IS NOT NULL
GROUP BY SalesPersonID;

Commands completed successfully.

(17 rows affected)

Total execution time: 00:00:00.028

Identification Number of the Salesperson,Personal Sales,Percentage of the Sales
284,2608116.3755,2.873151784863419
278,4069422.2109,4.482954747894905
281,7259567.8761,7.997281331648868
275,10475367.0751,11.539868347766165
276,11695019.0605,12.883460724119550
287,826417.4667,0.910397573435047
279,8086073.6761,8.907776214767233
290,5087977.212,5.605014770554361
282,6683536.6583,7.362714125558673
288,2062393.1371,2.271972439041546


# **UNIT IV:** **Unpivoting Quarterly Sales**

**Why should this proposition and the query be considered special?**

- It simplifies cross-product comparisons of quarterly sales by converting column-based data (Q1–Q4) into rows for cleaner, more intuitive analysis.

In [4]:
USE AdventureWorks2017;
GO

SELECT ProductID AS "Identification Number of the Product", Quarter, Sales
FROM (SELECT ProductID, SUM(CASE WHEN DATEPART(QUARTER, OrderDate) = 1 THEN LineTotal END) AS Q1, SUM(CASE WHEN DATEPART(QUARTER, OrderDate) = 2 THEN LineTotal END) AS Q2, SUM(CASE WHEN DATEPART(QUARTER, OrderDate) = 3 THEN LineTotal END) AS Q3, SUM(CASE WHEN DATEPART(QUARTER, OrderDate) = 4 THEN LineTotal END) AS Q4 FROM Sales.SalesOrderDetail JOIN Sales.SalesOrderHeader ON SalesOrderDetail.SalesOrderID = SalesOrderHeader.SalesOrderID GROUP BY ProductID) AS p
UNPIVOT(Sales FOR Quarter IN (Q1, Q2, Q3, Q4)) AS unpvt;

Commands completed successfully.

Warning: Null value is eliminated by an aggregate or other SET operation.

(1021 rows affected)

Total execution time: 00:00:00.209

Identification Number of the Product,Quarter,Sales
925,Q1,19633.494000
925,Q2,28476.060000
925,Q3,22843.894996
925,Q4,22630.974000
902,Q2,5201.352000
902,Q3,2000.520000
710,Q1,114.000000
710,Q2,51.300000
710,Q3,182.400000
710,Q4,165.300000


# **UNIT V:** **Quarterly Sales Rollup**

**Why should this proposition and the query be considered special?**

- It creates a structured report that combines monthly details with quarterly and yearly summaries, automatically adding subtotals for comprehensive insights.

In [5]:
USE AdventureWorks2017;
GO

SELECT YEAR(OrderDate) AS "Year", DATEPART(QUARTER, OrderDate) AS Quarter, MONTH(OrderDate) AS "Month", SUM(TotalDue) AS Sales
FROM Sales.SalesOrderHeader
GROUP BY ROLLUP(YEAR(OrderDate), DATEPART(QUARTER, OrderDate), MONTH(OrderDate));

Commands completed successfully.

(56 rows affected)

Total execution time: 00:00:00.035

Year,Quarter,Month,Sales
2011,2,5,567020.9498
2011,2,6,507096.469
2011,2,NULL,1074117.4188
2011,3,7,2292182.8828
2011,3,8,2800576.1723
2011,3,9,554791.6082
2011,3,NULL,5647550.6633
2011,4,10,5156269.5291
2011,4,11,815313.0152
2011,4,12,1462448.8986


# **UNIT VI:** **First Purchase Date per Customer**

**Why should this proposition and the query be considered special?**

- It analyzes customer loyalty patterns over time by identifying the timing of customers’ first purchases.

In [6]:
USE AdventureWorks2017;
GO

SELECT CustomerID AS "Identification Number of the Customer", FIRST_VALUE(OrderDate) OVER(PARTITION BY CustomerID ORDER BY OrderDate) AS "Date of the First Purchase" 
FROM Sales.SalesOrderHeader;

Commands completed successfully.

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.372

Identification Number of the Customer,Date of the First Purchase
11000,2011-06-21 00:00:00.000
11000,2011-06-21 00:00:00.000
11000,2011-06-21 00:00:00.000
11001,2011-06-17 00:00:00.000
11001,2011-06-17 00:00:00.000
11001,2011-06-17 00:00:00.000
11002,2011-06-09 00:00:00.000
11002,2011-06-09 00:00:00.000
11002,2011-06-09 00:00:00.000
11003,2011-05-31 00:00:00.000


# **UNIT VII:** **Product Price Quartiles**

**Why should this proposition and the query be considered special?**

- It reveals pricing strategies (e.g., budget vs. premium products) by grouping products into four equal price tiers using the NTILE function.

In [7]:
USE AdventureWorks2017;
GO

SELECT ProductID AS "Identification Number of the Product", Name AS "Name of the Product", ListPrice AS "Price", NTILE(4) OVER(ORDER BY ListPrice) AS "Price Quartile"
FROM Production.Product
WHERE ListPrice > 0;

Commands completed successfully.

(304 rows affected)

Total execution time: 00:00:00.017

Identification Number of the Product,Name of the Product,Price,Price Quartile
873,Patch Kit/8 Patches,2.29,1
922,Road Tire Tube,3.99,1
923,Touring Tire Tube,4.99,1
921,Mountain Tire Tube,4.99,1
870,Water Bottle - 30 oz.,4.99,1
877,Bike Wash - Dissolver,7.95,1
872,Road Bottle Cage,8.99,1
874,"Racing Socks, M",8.99,1
875,"Racing Socks, L",8.99,1
712,AWC Logo Cap,8.99,1


# **UNIT VIII:** **Year-over-Year Sales Growth**

**Why should this proposition and the query be considered special?**

- It highlights unbiased long-term growth trends by comparing current sales to the same month in the previous year (using a 12-month lag), minimizing seasonal distortions.

In [8]:
USE AdventureWorks2017;
GO

SELECT YEAR(OrderDate) AS "Current Year", MONTH(OrderDate) AS Month, SUM(TotalDue) AS "Monthly Sales", LAG(SUM(TotalDue), 12) OVER(ORDER BY YEAR(OrderDate), MONTH(OrderDate)) AS "Sales from the Previous Year"
FROM Sales.SalesOrderHeader
GROUP BY YEAR(OrderDate), MONTH(OrderDate);

Commands completed successfully.

(38 rows affected)

Total execution time: 00:00:00.039

Current Year,Month,Monthly Sales,Sales from the Previous Year
2011,5,567020.9498,NULL
2011,6,507096.469,NULL
2011,7,2292182.8828,NULL
2011,8,2800576.1723,NULL
2011,9,554791.6082,NULL
2011,10,5156269.5291,NULL
2011,11,815313.0152,NULL
2011,12,1462448.8986,NULL
2012,1,4458337.4444,NULL
2012,2,1649051.9001,NULL


# **UNIT IX:** **3-Month Sales Moving Average**

**Why should this proposition and the query be considered special?**

- It clarifies trends by reducing monthly fluctuations through a 3-month rolling average calculation.

In [9]:
USE AdventureWorks2017;
GO

SELECT YEAR(OrderDate) AS Year, MONTH(OrderDate) AS Month, SUM(TotalDue) AS "Monthly Sales", AVG(SUM(TotalDue)) OVER(ORDER BY YEAR(OrderDate), MONTH(OrderDate) ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS "Moving Average"
FROM Sales.SalesOrderHeader
GROUP BY YEAR(OrderDate), MONTH(OrderDate);

Commands completed successfully.

(38 rows affected)

Total execution time: 00:00:00.048

Year,Month,Monthly Sales,Moving Average
2011,5,567020.9498,567020.9498
2011,6,507096.469,537058.7094
2011,7,2292182.8828,1122100.1005
2011,8,2800576.1723,1866618.508
2011,9,554791.6082,1882516.8877
2011,10,5156269.5291,2837212.4365
2011,11,815313.0152,2175458.0508
2011,12,1462448.8986,2478010.4809
2012,1,4458337.4444,2245366.4527
2012,2,1649051.9001,2523279.4143


# **UNIT X:** **Customer Spending Tiers**

**Why should this proposition and the query be considered special?**

- It enables targeted loyalty programs by ranking customers into four spending tiers (Bronze to Platinum) based on their value.

In [10]:
USE AdventureWorks2017;
GO

SELECT CustomerID AS "Identification Number of the Customer", SUM(TotalDue) AS "Total Spent", NTILE(4) OVER(ORDER BY SUM(TotalDue)) AS "Spending Tier"
FROM Sales.SalesOrderHeader
GROUP BY CustomerID;

Commands completed successfully.

(19119 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.336

Identification Number of the Customer,Total Spent,Spending Tier
30078,1.5183,1
28968,2.5305,1
28300,2.5305,1
27991,2.5305,1
27993,2.5305,1
28781,2.5305,1
28094,2.5305,1
28095,2.5305,1
27992,2.5305,1
28093,2.5305,1
